In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('customer_shopping_behavior.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [4]:
df.describe(include = 'all')

df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [5]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [6]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [31]:
df.columns = df.columns.str.lower()     # To looks all columns name neat and consistency.
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [32]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [33]:
# Create a column age_group
labels = ['young Adult', 'Adult', 'Middle_aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [34]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle_aged
1,19,young Adult
2,50,Middle_aged
3,21,young Adult
4,45,Middle_aged
5,46,Middle_aged
6,63,Senior
7,27,young Adult
8,26,young Adult
9,57,Middle_aged


In [35]:
# Create column purchase_frequency_days  # used df['frequency_of_purchase'].unique() to see all the names in the column.

frequency_mapping = {
    'Fortnightly' : 14,
    'Weekly' : 7,
    'Annually' : 365,
    'Quarterly' : 90,
    'Bi-Weekly' : 14,
    'Monthly' : 30,
    'Every 3 Months' : 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [36]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [37]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [38]:
(df['discount_applied'] == df['promo_code_used']).all()  # Both the columns give the same values so drop the one column.

np.True_

In [39]:
df = df.drop('promo_code_used', axis =1)

In [40]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [41]:
%pip install psycopg2-binary sqlalchemy 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [42]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ==========================================
# 2. POSTGRESQL CONNECTION
# ==========================================

username = "postgres"
password = "Dhruvi@2306"
host = "localhost"
port = 5432
database = "customer_behaviour"

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

engine = create_engine(connection_url)


# ==========================================
# 3. UPLOAD DATA
# ==========================================

df.to_sql(
    name="customer",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False
)

print("\nData uploaded successfully!")


# ==========================================
# 4. CHECK TABLE FROM PYTHON
# ==========================================

with engine.connect() as conn:

    # Check number of rows
    result = conn.execute(
        text("SELECT COUNT(*) FROM public.customer")
    )

    row_count = result.scalar()

    print("\nRows in PostgreSQL:", row_count)


    # Check columns
    result = conn.execute(
        text("""
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
            AND table_name = 'customer'
            ORDER BY ordinal_position
        """)
    )

    print("\nColumns in PostgreSQL:")

    for row in result:
        print(row)


    # Display data
    result = conn.execute(
        text("SELECT * FROM public.customer LIMIT 5")
    )

    print("\nData inside PostgreSQL:")

    for row in result:
        print(row)



Data uploaded successfully!

Rows in PostgreSQL: 3900

Columns in PostgreSQL:
('customer_id', 'bigint')
('age', 'bigint')
('gender', 'text')
('item_purchased', 'text')
('category', 'text')
('purchase_amount', 'bigint')
('location', 'text')
('size', 'text')
('color', 'text')
('season', 'text')
('review_rating', 'double precision')
('subscription_status', 'text')
('shipping_type', 'text')
('discount_applied', 'text')
('previous_purchases', 'bigint')
('payment_method', 'text')
('frequency_of_purchases', 'text')
('age_group', 'text')
('purchase_frequency_days', 'bigint')

Data inside PostgreSQL:
(1, 55, 'Male', 'Blouse', 'Clothing', 53, 'Kentucky', 'L', 'Gray', 'Winter', 3.1, 'Yes', 'Express', 'Yes', 14, 'Venmo', 'Fortnightly', 'Middle_aged', 14)
(2, 19, 'Male', 'Sweater', 'Clothing', 64, 'Maine', 'L', 'Maroon', 'Winter', 3.1, 'Yes', 'Express', 'Yes', 2, 'Cash', 'Fortnightly', 'young Adult', 14)
(3, 50, 'Male', 'Jeans', 'Clothing', 73, 'Massachusetts', 'S', 'Maroon', 'Spring', 3.1, 'Yes',